In [1]:
# ============================================================
# CELL 1 - IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# ============================================================
# CELL 2 - LOAD DATASET
# ============================================================

from pathlib import Path

possible_paths = [
    Path("../data/nyc_taxi_trip_duration.csv"),
    Path("data/nyc_taxi_trip_duration.csv"),
    Path("./nyc_taxi_trip_duration.csv"),
]

file_path = None

for path in possible_paths:
    if path.exists():
        file_path = path
        break

if file_path is None:
    raise FileNotFoundError(
        "nyc_taxi_trip_duration.csv not found. "
        "Place it inside ../data/, data/, or the same folder as this notebook."
    )

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("File used:", file_path)
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully.
File used: ..\data\nyc_taxi_trip_duration.csv
Shape: (729322, 11)


,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id1080784,2,2016-02-29 16:40:21,2016-02-29 16:47:01,1,-73.953918,40.778873,-73.963875,40.771164,N,400
1,id0889885,1,2016-03-11 23:35:37,2016-03-11 23:53:57,2,-73.988312,40.731743,-73.994751,40.694931,N,1100
2,id0857912,2,2016-02-21 17:59:33,2016-02-21 18:26:48,2,-73.997314,40.721458,-73.948029,40.774918,N,1635
3,id3744273,2,2016-01-05 09:44:31,2016-01-05 10:03:32,6,-73.961670,40.759720,-73.956779,40.780628,N,1141
4,id0232939,1,2016-02-17 06:42:23,2016-02-17 06:56:31,1,-74.017120,40.708469,-73.988182,40.740631,N,848


In [3]:
# ============================================================
# CELL 3 - CONVERT DATETIME COLUMNS
# ============================================================

df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
df["dropoff_datetime"] = pd.to_datetime(df["dropoff_datetime"])

print("Datetime conversion completed.")

display(df[["pickup_datetime", "dropoff_datetime"]].dtypes)

Datetime conversion completed.


pickup_datetime     datetime64[us]
dropoff_datetime    datetime64[us]
dtype: object

In [4]:
# ============================================================
# CELL 4 - BASIC DATASET CHECK
# ============================================================

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes)

print("\nMissing Values:")
display(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Shape: (729322, 11)

Columns:
['id', 'vendor_id', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'store_and_fwd_flag', 'trip_duration']

Data Types:


id                               str
vendor_id                      int64
pickup_datetime       datetime64[us]
dropoff_datetime      datetime64[us]
passenger_count                int64
pickup_longitude             float64
pickup_latitude              float64
dropoff_longitude            float64
dropoff_latitude             float64
store_and_fwd_flag               str
trip_duration                  int64
dtype: object


Missing Values:


id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64


Duplicate Rows: 0


In [5]:
# ============================================================
# CELL 5 - CREATE WORKING COPY
# ============================================================

fe_df = df.copy()

print("Working copy created successfully.")
print("Shape:", fe_df.shape)

Working copy created successfully.
Shape: (729322, 11)


In [6]:
# ============================================================
# CELL 5 - CREATE WORKING COPY
# ============================================================

fe_df = df.copy()

print("Working copy created successfully.")
print("Shape:", fe_df.shape)

Working copy created successfully.
Shape: (729322, 11)


In [7]:
# ============================================================
# CELL 6 - MISSING VALUE HANDLING
# ============================================================

missing_summary = pd.DataFrame({
    "Missing Values": fe_df.isnull().sum(),
    "Missing Percentage": (fe_df.isnull().sum() / len(fe_df)) * 100
})

display(missing_summary)

print("No missing value treatment applied if missing values are zero.")

,Missing Values,Missing Percentage
id,0,0.0
vendor_id,0,0.0
pickup_datetime,0,0.0
dropoff_datetime,0,0.0
passenger_count,0,0.0
pickup_longitude,0,0.0
pickup_latitude,0,0.0
dropoff_longitude,0,0.0
dropoff_latitude,0,0.0
store_and_fwd_flag,0,0.0


No missing value treatment applied if missing values are zero.


In [8]:
# ============================================================
# CELL 7 - INVALID VALUE HANDLING
# ============================================================

print("Invalid passenger_count rows:", fe_df[fe_df["passenger_count"] <= 0].shape[0])
print("Invalid trip_duration rows:", fe_df[fe_df["trip_duration"] <= 0].shape[0])

before_shape = fe_df.shape

fe_df = fe_df[
    (fe_df["passenger_count"] > 0) &
    (fe_df["trip_duration"] > 0)
]

after_shape = fe_df.shape

print("\nShape before:", before_shape)
print("Shape after:", after_shape)

Invalid passenger_count rows: 33
Invalid trip_duration rows: 0

Shape before: (729322, 11)
Shape after: (729289, 11)


In [9]:
# ============================================================
# CELL 8 - TARGET AND FEATURE SEPARATION
# ============================================================

target = "trip_duration"

X = fe_df.drop(columns=[target])
y = fe_df[target]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (729289, 10)
Target shape: (729289,)


In [10]:
# ============================================================
# CELL 9 - TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (583431, 10)
X_test shape: (145858, 10)
y_train shape: (583431,)
y_test shape: (145858,)


In [11]:
# ============================================================
# CELL 10 - CREATE TRAIN AND TEST COPIES
# ============================================================

train_df = X_train.copy()
test_df = X_test.copy()

train_df["trip_duration"] = y_train
test_df["trip_duration"] = y_test

print("Train copy shape:", train_df.shape)
print("Test copy shape:", test_df.shape)

Train copy shape: (583431, 11)
Test copy shape: (145858, 11)


In [12]:
# ============================================================
# CELL 11 - DATETIME FEATURE CREATION
# ============================================================

def create_datetime_features(data):
    data = data.copy()

    data["pickup_hour"] = data["pickup_datetime"].dt.hour
    data["pickup_day"] = data["pickup_datetime"].dt.day
    data["pickup_month"] = data["pickup_datetime"].dt.month
    data["pickup_year"] = data["pickup_datetime"].dt.year
    data["pickup_dayofweek"] = data["pickup_datetime"].dt.dayofweek
    data["pickup_day_name"] = data["pickup_datetime"].dt.day_name()

    return data

train_df = create_datetime_features(train_df)
test_df = create_datetime_features(test_df)

print("Datetime features created successfully.")

display(train_df[[
    "pickup_datetime",
    "pickup_hour",
    "pickup_day",
    "pickup_month",
    "pickup_year",
    "pickup_dayofweek",
    "pickup_day_name"
]].head())

Datetime features created successfully.


,pickup_datetime,pickup_hour,pickup_day,pickup_month,pickup_year,pickup_dayofweek,pickup_day_name
199266,2016-01-01 18:37:35,18,1,1,2016,4,Friday
278684,2016-05-31 16:35:44,16,31,5,2016,1,Tuesday
34691,2016-03-26 16:09:19,16,26,3,2016,5,Saturday
378162,2016-04-07 15:26:36,15,7,4,2016,3,Thursday
249800,2016-04-16 03:24:50,3,16,4,2016,5,Saturday


In [13]:
# ============================================================
# CELL 12 - WEEKEND AND RUSH HOUR FEATURES
# ============================================================

def create_time_flags(data):
    data = data.copy()

    data["is_weekend"] = data["pickup_dayofweek"].apply(
        lambda x: 1 if x in [5, 6] else 0
    )

    data["is_rush_hour"] = data["pickup_hour"].apply(
        lambda x: 1 if (7 <= x <= 10) or (16 <= x <= 20) else 0
    )

    return data

train_df = create_time_flags(train_df)
test_df = create_time_flags(test_df)

print("Weekend and rush-hour features created successfully.")

display(train_df[[
    "pickup_day_name",
    "pickup_hour",
    "is_weekend",
    "is_rush_hour"
]].head())

Weekend and rush-hour features created successfully.


,pickup_day_name,pickup_hour,is_weekend,is_rush_hour
199266,Friday,18,0,1
278684,Tuesday,16,0,1
34691,Saturday,16,1,1
378162,Thursday,15,0,0
249800,Saturday,3,1,0


In [14]:
# ============================================================
# CELL 12.1 - OFFICE HOUR FEATURE
# ============================================================

train_df["is_office_hour"] = train_df["pickup_hour"].apply(
    lambda x: 1 if 9 <= x <= 18 else 0
)

test_df["is_office_hour"] = test_df["pickup_hour"].apply(
    lambda x: 1 if 9 <= x <= 18 else 0
)

print("Office hour feature created successfully.")

display(train_df[["pickup_hour", "is_rush_hour", "is_office_hour"]].head())

Office hour feature created successfully.


,pickup_hour,is_rush_hour,is_office_hour
199266,18,1,1
278684,16,1,1
34691,16,1,1
378162,15,0,1
249800,3,0,0


In [15]:
# ============================================================
# CELL 13 - TIME SLOT FEATURE
# ============================================================

def create_time_slot(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

train_df["time_slot"] = train_df["pickup_hour"].apply(create_time_slot)
test_df["time_slot"] = test_df["pickup_hour"].apply(create_time_slot)

print("Time slot feature created successfully.")

display(train_df[["pickup_hour", "time_slot"]].head())

Time slot feature created successfully.


,pickup_hour,time_slot
199266,18,Evening
278684,16,Afternoon
34691,16,Afternoon
378162,15,Afternoon
249800,3,Night


In [16]:
# ============================================================
# CELL 14 - PASSENGER GROUP FEATURE
# ============================================================

def create_passenger_group(count):
    if count == 1:
        return "Solo"
    elif count == 2:
        return "Couple"
    elif 3 <= count <= 4:
        return "Small Group"
    else:
        return "Large Group"

train_df["passenger_group"] = train_df["passenger_count"].apply(create_passenger_group)
test_df["passenger_group"] = test_df["passenger_count"].apply(create_passenger_group)

print("Passenger group feature created successfully.")

display(train_df[["passenger_count", "passenger_group"]].head())

Passenger group feature created successfully.


,passenger_count,passenger_group
199266,1,Solo
278684,2,Couple
34691,1,Solo
378162,2,Couple
249800,1,Solo


In [17]:
# ============================================================
# CELL 15 - HAVERSINE DISTANCE FEATURE
# ============================================================

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c

train_df["trip_distance_km"] = haversine_distance(
    train_df["pickup_latitude"],
    train_df["pickup_longitude"],
    train_df["dropoff_latitude"],
    train_df["dropoff_longitude"]
)

test_df["trip_distance_km"] = haversine_distance(
    test_df["pickup_latitude"],
    test_df["pickup_longitude"],
    test_df["dropoff_latitude"],
    test_df["dropoff_longitude"]
)

print("Trip distance feature created successfully.")

display(train_df[["trip_distance_km"]].head())

Trip distance feature created successfully.


,trip_distance_km
199266,2.520748
278684,2.287900
34691,1.736363
378162,1.420827
249800,1.067475


In [18]:
# ============================================================
# CELL 15.1 - ADVANCED DISTANCE FEATURES
# ============================================================

print("ADVANCED DISTANCE FEATURES")
print("=" * 70)

# Coordinate differences
train_df["lat_diff"] = train_df["dropoff_latitude"] - train_df["pickup_latitude"]
train_df["lon_diff"] = train_df["dropoff_longitude"] - train_df["pickup_longitude"]

test_df["lat_diff"] = test_df["dropoff_latitude"] - test_df["pickup_latitude"]
test_df["lon_diff"] = test_df["dropoff_longitude"] - test_df["pickup_longitude"]

# Manhattan-style distance approximation
# Useful for NYC because roads follow a grid-like structure
train_df["manhattan_distance_km"] = (
    np.abs(train_df["lat_diff"]) + np.abs(train_df["lon_diff"])
) * 111

test_df["manhattan_distance_km"] = (
    np.abs(test_df["lat_diff"]) + np.abs(test_df["lon_diff"])
) * 111

# Direction angle from pickup to dropoff
train_df["direction_angle"] = np.degrees(
    np.arctan2(train_df["lat_diff"], train_df["lon_diff"])
)

test_df["direction_angle"] = np.degrees(
    np.arctan2(test_df["lat_diff"], test_df["lon_diff"])
)

advanced_distance_features = [
    "trip_distance_km",
    "lat_diff",
    "lon_diff",
    "manhattan_distance_km",
    "direction_angle"
]

display(train_df[advanced_distance_features].head())

print("\nAdvanced distance features created successfully.")

ADVANCED DISTANCE FEATURES


,trip_distance_km,lat_diff,lon_diff,manhattan_distance_km,direction_angle
199266,2.520748,0.020939,0.011467,3.597050,61.293091
278684,2.287900,0.009853,0.023849,3.741016,22.447859
34691,1.736363,-0.010059,0.015762,2.866207,-32.545601
378162,1.420827,0.012218,0.004936,1.904171,68.001476
249800,1.067475,-0.009598,0.000275,1.095840,-88.360823



Advanced distance features created successfully.


In [19]:
# ============================================================
# CELL 15.2 - BETTER AIRPORT FEATURES
# ============================================================

print("BETTER AIRPORT FEATURES")
print("=" * 70)

# Approximate coordinates of major NYC airports
airport_locations = {
    "JFK": (40.6413, -73.7781),
    "LGA": (40.7769, -73.8740),
    "EWR": (40.6895, -74.1745)
}

def min_airport_distance(lat, lon):
    """
    Calculates minimum Haversine distance from a point to NYC major airports.
    """
    distances = []
    
    for airport, (airport_lat, airport_lon) in airport_locations.items():
        distance = haversine_distance(
            lat,
            lon,
            airport_lat,
            airport_lon
        )
        distances.append(distance)
    
    return min(distances)

# Minimum distance from pickup point to nearest airport
train_df["pickup_airport_distance_km"] = train_df.apply(
    lambda row: min_airport_distance(
        row["pickup_latitude"],
        row["pickup_longitude"]
    ),
    axis=1
)

test_df["pickup_airport_distance_km"] = test_df.apply(
    lambda row: min_airport_distance(
        row["pickup_latitude"],
        row["pickup_longitude"]
    ),
    axis=1
)

# Minimum distance from dropoff point to nearest airport
train_df["dropoff_airport_distance_km"] = train_df.apply(
    lambda row: min_airport_distance(
        row["dropoff_latitude"],
        row["dropoff_longitude"]
    ),
    axis=1
)

test_df["dropoff_airport_distance_km"] = test_df.apply(
    lambda row: min_airport_distance(
        row["dropoff_latitude"],
        row["dropoff_longitude"]
    ),
    axis=1
)

# Airport flags
airport_threshold_km = 3

train_df["is_pickup_near_airport"] = (
    train_df["pickup_airport_distance_km"] <= airport_threshold_km
).astype(int)

test_df["is_pickup_near_airport"] = (
    test_df["pickup_airport_distance_km"] <= airport_threshold_km
).astype(int)

train_df["is_dropoff_near_airport"] = (
    train_df["dropoff_airport_distance_km"] <= airport_threshold_km
).astype(int)

test_df["is_dropoff_near_airport"] = (
    test_df["dropoff_airport_distance_km"] <= airport_threshold_km
).astype(int)

# Airport trip type
def airport_trip_type(row):
    if row["is_pickup_near_airport"] == 1 and row["is_dropoff_near_airport"] == 1:
        return "airport_to_airport"
    elif row["is_pickup_near_airport"] == 1:
        return "airport_pickup"
    elif row["is_dropoff_near_airport"] == 1:
        return "airport_dropoff"
    else:
        return "non_airport_trip"

train_df["airport_trip_type"] = train_df.apply(airport_trip_type, axis=1)
test_df["airport_trip_type"] = test_df.apply(airport_trip_type, axis=1)

airport_features = [
    "pickup_airport_distance_km",
    "dropoff_airport_distance_km",
    "is_pickup_near_airport",
    "is_dropoff_near_airport",
    "airport_trip_type"
]

display(train_df[airport_features].head())

print("\nAirport feature distribution:")
display(train_df["airport_trip_type"].value_counts())

print("\nBetter airport features created successfully.")

BETTER AIRPORT FEATURES


,pickup_airport_distance_km,dropoff_airport_distance_km,is_pickup_near_airport,is_dropoff_near_airport,airport_trip_type
199266,10.865796,9.132218,0,0,non_airport_trip
278684,8.292446,6.146941,0,0,non_airport_trip
34691,10.655630,9.986953,0,0,non_airport_trip
378162,9.985379,9.265138,0,0,non_airport_trip
249800,9.161230,10.163203,0,0,non_airport_trip



Airport feature distribution:


airport_trip_type
non_airport_trip      542226
airport_pickup         26404
airport_dropoff        13146
airport_to_airport      1655
Name: count, dtype: int64


Better airport features created successfully.


In [20]:
# ============================================================
# CELL 16 - DURATION AND SPEED FEATURES
# ============================================================

train_df["trip_duration_minutes"] = train_df["trip_duration"] / 60
test_df["trip_duration_minutes"] = test_df["trip_duration"] / 60

train_df["trip_duration_log"] = np.log1p(train_df["trip_duration"])
test_df["trip_duration_log"] = np.log1p(test_df["trip_duration"])

train_df["average_speed_kmh"] = (
    train_df["trip_distance_km"] / (train_df["trip_duration"] / 3600)
)

test_df["average_speed_kmh"] = (
    test_df["trip_distance_km"] / (test_df["trip_duration"] / 3600)
)

print("Duration and speed features created successfully.")

display(train_df[[
    "trip_duration",
    "trip_duration_minutes",
    "trip_duration_log",
    "trip_distance_km",
    "average_speed_kmh"
]].head())

Duration and speed features created successfully.


,trip_duration,trip_duration_minutes,trip_duration_log,trip_distance_km,average_speed_kmh
199266,526,8.766667,6.267201,2.520748,17.252268
278684,1085,18.083333,6.990257,2.287900,7.591190
34691,442,7.366667,6.093570,1.736363,14.142325
378162,678,11.300000,6.520621,1.420827,7.544212
249800,323,5.383333,5.780744,1.067475,11.897559


In [21]:
# ============================================================
# CELL 17 - SPEED CATEGORY FEATURE
# ============================================================

def create_speed_category(speed):
    if speed < 10:
        return "Slow"
    elif speed < 30:
        return "Medium"
    else:
        return "Fast"

train_df["speed_category"] = train_df["average_speed_kmh"].apply(create_speed_category)
test_df["speed_category"] = test_df["average_speed_kmh"].apply(create_speed_category)

print("Speed category feature created successfully.")

display(train_df[["average_speed_kmh", "speed_category"]].head())

Speed category feature created successfully.


,average_speed_kmh,speed_category
199266,17.252268,Medium
278684,7.591190,Slow
34691,14.142325,Medium
378162,7.544212,Slow
249800,11.897559,Medium


In [22]:
# ============================================================
# CELL 17.1 - BETTER TIME-BASED FEATURES
# ============================================================

print("BETTER TIME-BASED FEATURES")
print("=" * 70)

# Morning peak: common office commute time
train_df["is_morning_peak"] = train_df["pickup_hour"].between(7, 10).astype(int)
test_df["is_morning_peak"] = test_df["pickup_hour"].between(7, 10).astype(int)

# Evening peak: return commute time
train_df["is_evening_peak"] = train_df["pickup_hour"].between(16, 20).astype(int)
test_df["is_evening_peak"] = test_df["pickup_hour"].between(16, 20).astype(int)

# Late night trips
train_df["is_late_night"] = train_df["pickup_hour"].apply(
    lambda x: 1 if (x >= 22) or (x <= 5) else 0
)

test_df["is_late_night"] = test_df["pickup_hour"].apply(
    lambda x: 1 if (x >= 22) or (x <= 5) else 0
)

# Business day: Monday to Friday
train_df["is_business_day"] = (train_df["pickup_dayofweek"] < 5).astype(int)
test_df["is_business_day"] = (test_df["pickup_dayofweek"] < 5).astype(int)

# Month start and month end indicators
train_df["is_month_start"] = train_df["pickup_datetime"].dt.is_month_start.astype(int)
test_df["is_month_start"] = test_df["pickup_datetime"].dt.is_month_start.astype(int)

train_df["is_month_end"] = train_df["pickup_datetime"].dt.is_month_end.astype(int)
test_df["is_month_end"] = test_df["pickup_datetime"].dt.is_month_end.astype(int)

better_time_features = [
    "pickup_hour",
    "pickup_dayofweek",
    "is_morning_peak",
    "is_evening_peak",
    "is_late_night",
    "is_business_day",
    "is_month_start",
    "is_month_end"
]

display(train_df[better_time_features].head())

print("\nBetter time-based features created successfully.")

BETTER TIME-BASED FEATURES


,pickup_hour,pickup_dayofweek,is_morning_peak,is_evening_peak,is_late_night,is_business_day,is_month_start,is_month_end
199266,18,4,0,1,0,1,1,0
278684,16,1,0,1,0,1,0,1
34691,16,5,0,1,0,0,0,0
378162,15,3,0,0,0,1,0,0
249800,3,5,0,0,1,0,0,0



Better time-based features created successfully.


In [23]:
# ============================================================
# CELL 18 - INTERACTION FEATURES
# ============================================================

print("INTERACTION FEATURES")
print("=" * 70)

# Distance with rush hour
train_df["distance_rush_interaction"] = (
    train_df["trip_distance_km"] * train_df["is_rush_hour"]
)

test_df["distance_rush_interaction"] = (
    test_df["trip_distance_km"] * test_df["is_rush_hour"]
)

# Passenger count with distance
train_df["passenger_distance_interaction"] = (
    train_df["passenger_count"] * train_df["trip_distance_km"]
)

test_df["passenger_distance_interaction"] = (
    test_df["passenger_count"] * test_df["trip_distance_km"]
)

# Distance with late night
train_df["distance_late_night_interaction"] = (
    train_df["trip_distance_km"] * train_df["is_late_night"]
)

test_df["distance_late_night_interaction"] = (
    test_df["trip_distance_km"] * test_df["is_late_night"]
)

# Distance with weekend
train_df["distance_weekend_interaction"] = (
    train_df["trip_distance_km"] * train_df["is_weekend"]
)

test_df["distance_weekend_interaction"] = (
    test_df["trip_distance_km"] * test_df["is_weekend"]
)

# Airport with rush hour
train_df["airport_rush_hour_interaction"] = (
    train_df["is_pickup_near_airport"] * train_df["is_rush_hour"]
)

test_df["airport_rush_hour_interaction"] = (
    test_df["is_pickup_near_airport"] * test_df["is_rush_hour"]
)

interaction_features = [
    "distance_rush_interaction",
    "passenger_distance_interaction",
    "distance_late_night_interaction",
    "distance_weekend_interaction",
    "airport_rush_hour_interaction"
]

display(train_df[interaction_features].head())

print("\nInteraction features created successfully.")

INTERACTION FEATURES


,distance_rush_interaction,passenger_distance_interaction,distance_late_night_interaction,distance_weekend_interaction,airport_rush_hour_interaction
199266,2.520748,2.520748,0.000000,0.000000,0
278684,2.287900,4.575801,0.000000,0.000000,0
34691,1.736363,1.736363,0.000000,1.736363,0
378162,0.000000,2.841653,0.000000,0.000000,0
249800,0.000000,1.067475,1.067475,1.067475,0



Interaction features created successfully.


In [24]:
# ============================================================
# CELL 18.1 - LOCATION CLUSTERING FEATURES
# ============================================================

print("LOCATION CLUSTERING FEATURES")
print("=" * 70)

from sklearn.cluster import MiniBatchKMeans

# Number of location clusters
# 20 is a good starting point for NYC taxi data
n_clusters = 20

# Pickup location clustering
pickup_kmeans = MiniBatchKMeans(
    n_clusters=n_clusters,
    random_state=42,
    batch_size=10000
)

train_df["pickup_cluster"] = pickup_kmeans.fit_predict(
    train_df[["pickup_latitude", "pickup_longitude"]]
)

test_df["pickup_cluster"] = pickup_kmeans.predict(
    test_df[["pickup_latitude", "pickup_longitude"]]
)

# Dropoff location clustering
dropoff_kmeans = MiniBatchKMeans(
    n_clusters=n_clusters,
    random_state=42,
    batch_size=10000
)

train_df["dropoff_cluster"] = dropoff_kmeans.fit_predict(
    train_df[["dropoff_latitude", "dropoff_longitude"]]
)

test_df["dropoff_cluster"] = dropoff_kmeans.predict(
    test_df[["dropoff_latitude", "dropoff_longitude"]]
)

cluster_features = [
    "pickup_cluster",
    "dropoff_cluster"
]

display(train_df[cluster_features].head())

print("\nPickup cluster distribution:")
display(train_df["pickup_cluster"].value_counts().sort_index())

print("\nDropoff cluster distribution:")
display(train_df["dropoff_cluster"].value_counts().sort_index())

print("\nLocation clustering features created successfully.")

LOCATION CLUSTERING FEATURES


,pickup_cluster,dropoff_cluster
199266,14,9
278684,18,11
34691,14,16
378162,10,2
249800,2,0



Pickup cluster distribution:


pickup_cluster
0     39656
1     39984
2      4971
3     25666
4     32378
5     13222
6     48864
7     21603
8     15938
9     26477
10    32028
11    13124
12     9263
13    36227
14    43750
15    33326
16    39032
17    33809
18    36292
19    37821
Name: count, dtype: int64


Dropoff cluster distribution:


dropoff_cluster
0     13568
1     38544
2     39738
3     47558
4      7161
5     29327
6     11858
7     20297
8     14382
9     45877
10    50618
11    44876
12     1348
13    51175
14    38432
15    33320
16    52315
17     4122
18     9440
19    29475
Name: count, dtype: int64


Location clustering features created successfully.


In [25]:
# ============================================================
# CELL 19 - AGGREGATION FEATURES
# ============================================================

print("AGGREGATION FEATURES")
print("=" * 70)

# Mean trip duration by pickup hour
avg_duration_by_hour = train_df.groupby("pickup_hour")["trip_duration"].mean()

train_df["avg_duration_by_hour"] = train_df["pickup_hour"].map(avg_duration_by_hour)
test_df["avg_duration_by_hour"] = test_df["pickup_hour"].map(avg_duration_by_hour)

# Mean trip duration by pickup day of week
avg_duration_by_dayofweek = train_df.groupby("pickup_dayofweek")["trip_duration"].mean()

train_df["avg_duration_by_dayofweek"] = train_df["pickup_dayofweek"].map(avg_duration_by_dayofweek)
test_df["avg_duration_by_dayofweek"] = test_df["pickup_dayofweek"].map(avg_duration_by_dayofweek)

# Mean trip duration by pickup month
avg_duration_by_month = train_df.groupby("pickup_month")["trip_duration"].mean()

train_df["avg_duration_by_month"] = train_df["pickup_month"].map(avg_duration_by_month)
test_df["avg_duration_by_month"] = test_df["pickup_month"].map(avg_duration_by_month)

# Mean trip duration by pickup cluster
avg_duration_by_pickup_cluster = train_df.groupby("pickup_cluster")["trip_duration"].mean()

train_df["avg_duration_by_pickup_cluster"] = train_df["pickup_cluster"].map(avg_duration_by_pickup_cluster)
test_df["avg_duration_by_pickup_cluster"] = test_df["pickup_cluster"].map(avg_duration_by_pickup_cluster)

# Mean trip duration by dropoff cluster
avg_duration_by_dropoff_cluster = train_df.groupby("dropoff_cluster")["trip_duration"].mean()

train_df["avg_duration_by_dropoff_cluster"] = train_df["dropoff_cluster"].map(avg_duration_by_dropoff_cluster)
test_df["avg_duration_by_dropoff_cluster"] = test_df["dropoff_cluster"].map(avg_duration_by_dropoff_cluster)

aggregation_features = [
    "avg_duration_by_hour",
    "avg_duration_by_dayofweek",
    "avg_duration_by_month",
    "avg_duration_by_pickup_cluster",
    "avg_duration_by_dropoff_cluster"
]

display(train_df[aggregation_features].head())

print("\nMissing values in aggregation features:")
display(train_df[aggregation_features].isnull().sum())

print("\nAggregation features created successfully.")

AGGREGATION FEATURES


,avg_duration_by_hour,avg_duration_by_dayofweek,avg_duration_by_month,avg_duration_by_pickup_cluster,avg_duration_by_dropoff_cluster
199266,983.861324,990.124618,934.867525,891.033143,874.391612
278684,1095.741322,983.213661,994.461525,885.020969,762.804483
34691,1095.741322,907.512959,928.642924,891.033143,810.189487
378162,1097.970935,1012.426531,962.681159,880.108624,988.785822
249800,921.371569,907.512959,962.681159,927.975860,1449.214991



Missing values in aggregation features:


avg_duration_by_hour               0
avg_duration_by_dayofweek          0
avg_duration_by_month              0
avg_duration_by_pickup_cluster     0
avg_duration_by_dropoff_cluster    0
dtype: int64


Aggregation features created successfully.


In [26]:
# ============================================================
# CELL 20 - ENCODING CATEGORICAL FEATURES
# ============================================================

print("ENCODING CATEGORICAL FEATURES")
print("=" * 70)

from sklearn.preprocessing import LabelEncoder

# Categorical columns created/available after feature engineering
categorical_cols = [
    "store_and_fwd_flag",
    "pickup_day_name",
    "time_slot",
    "distance_category",
    "airport_trip_type"
]

# Keep only columns that actually exist in both train and test
categorical_cols = [
    col for col in categorical_cols 
    if col in train_df.columns and col in test_df.columns
]

print("Categorical columns selected for encoding:")
print(categorical_cols)

# Label Encoding
# Fitting encoder on combined train + test categories prevents unseen-category error
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    
    combined_values = pd.concat([
        train_df[col].astype(str),
        test_df[col].astype(str)
    ], axis=0)
    
    le.fit(combined_values)
    
    train_df[col + "_encoded"] = le.transform(train_df[col].astype(str))
    test_df[col + "_encoded"] = le.transform(test_df[col].astype(str))
    
    label_encoders[col] = le

encoded_cols = [col + "_encoded" for col in categorical_cols]

display(train_df[encoded_cols].head())

print("\nEncoded categorical features created successfully.")

ENCODING CATEGORICAL FEATURES
Categorical columns selected for encoding:
['store_and_fwd_flag', 'pickup_day_name', 'time_slot', 'airport_trip_type']


,store_and_fwd_flag_encoded,pickup_day_name_encoded,time_slot_encoded,airport_trip_type_encoded
199266,0,0,1,3
278684,0,5,0,3
34691,0,2,0,3
378162,0,4,0,3
249800,0,2,3,3



Encoded categorical features created successfully.


In [27]:
# ============================================================
# CELL 21 - SCALING NUMERICAL FEATURES
# ============================================================

print("SCALING NUMERICAL FEATURES")
print("=" * 70)

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Numerical columns to scale
numerical_cols_to_scale = [
    "passenger_count",
    "trip_distance_km",
    "manhattan_distance_km",
    "lat_diff",
    "lon_diff",
    "direction_angle",
    "pickup_airport_distance_km",
    "dropoff_airport_distance_km",
    "distance_rush_interaction",
    "passenger_distance_interaction",
    "distance_late_night_interaction",
    "distance_weekend_interaction",
    "airport_rush_hour_interaction",
    "avg_duration_by_hour",
    "avg_duration_by_dayofweek",
    "avg_duration_by_month",
    "avg_duration_by_pickup_cluster",
    "avg_duration_by_dropoff_cluster"
]

# Keep only columns that exist in both train and test
numerical_cols_to_scale = [
    col for col in numerical_cols_to_scale
    if col in train_df.columns and col in test_df.columns
]

print("Numerical columns selected for scaling:")
print(numerical_cols_to_scale)

# Fill missing values before scaling
for col in numerical_cols_to_scale:
    train_median = train_df[col].median()
    train_df[col] = train_df[col].fillna(train_median)
    test_df[col] = test_df[col].fillna(train_median)

# StandardScaler
standard_scaler = StandardScaler()

train_standard_scaled = standard_scaler.fit_transform(train_df[numerical_cols_to_scale])
test_standard_scaled = standard_scaler.transform(test_df[numerical_cols_to_scale])

# MinMaxScaler
minmax_scaler = MinMaxScaler()

train_minmax_scaled = minmax_scaler.fit_transform(train_df[numerical_cols_to_scale])
test_minmax_scaled = minmax_scaler.transform(test_df[numerical_cols_to_scale])

# RobustScaler
robust_scaler = RobustScaler()

train_robust_scaled = robust_scaler.fit_transform(train_df[numerical_cols_to_scale])
test_robust_scaled = robust_scaler.transform(test_df[numerical_cols_to_scale])

# Create scaled DataFrames
train_standard_scaled_df = pd.DataFrame(
    train_standard_scaled,
    columns=[col + "_standard_scaled" for col in numerical_cols_to_scale],
    index=train_df.index
)

test_standard_scaled_df = pd.DataFrame(
    test_standard_scaled,
    columns=[col + "_standard_scaled" for col in numerical_cols_to_scale],
    index=test_df.index
)

train_minmax_scaled_df = pd.DataFrame(
    train_minmax_scaled,
    columns=[col + "_minmax_scaled" for col in numerical_cols_to_scale],
    index=train_df.index
)

test_minmax_scaled_df = pd.DataFrame(
    test_minmax_scaled,
    columns=[col + "_minmax_scaled" for col in numerical_cols_to_scale],
    index=test_df.index
)

train_robust_scaled_df = pd.DataFrame(
    train_robust_scaled,
    columns=[col + "_robust_scaled" for col in numerical_cols_to_scale],
    index=train_df.index
)

test_robust_scaled_df = pd.DataFrame(
    test_robust_scaled,
    columns=[col + "_robust_scaled" for col in numerical_cols_to_scale],
    index=test_df.index
)

display(train_robust_scaled_df.head())

print("\nScaling completed successfully.")
print("StandardScaler, MinMaxScaler, and RobustScaler outputs are ready.")

SCALING NUMERICAL FEATURES
Numerical columns selected for scaling:
['passenger_count', 'trip_distance_km', 'manhattan_distance_km', 'lat_diff', 'lon_diff', 'direction_angle', 'pickup_airport_distance_km', 'dropoff_airport_distance_km', 'distance_rush_interaction', 'passenger_distance_interaction', 'distance_late_night_interaction', 'distance_weekend_interaction', 'airport_rush_hour_interaction', 'avg_duration_by_hour', 'avg_duration_by_dayofweek', 'avg_duration_by_month', 'avg_duration_by_pickup_cluster', 'avg_duration_by_dropoff_cluster']


,passenger_count_robust_scaled,trip_distance_km_robust_scaled,manhattan_distance_km_robust_scaled,lat_diff_robust_scaled,lon_diff_robust_scaled,direction_angle_robust_scaled,pickup_airport_distance_km_robust_scaled,dropoff_airport_distance_km_robust_scaled,distance_rush_interaction_robust_scaled,passenger_distance_interaction_robust_scaled,distance_late_night_interaction_robust_scaled,distance_weekend_interaction_robust_scaled,airport_rush_hour_interaction_robust_scaled,avg_duration_by_hour_robust_scaled,avg_duration_by_dayofweek_robust_scaled,avg_duration_by_month_robust_scaled,avg_duration_by_pickup_cluster_robust_scaled,avg_duration_by_dropoff_cluster_robust_scaled
199266,0.0,0.160021,0.144992,0.754269,0.401453,0.338221,0.538485,-0.095600,1.406312,-0.056101,0.000000,0.000000,0.0,0.507382,0.190323,-0.422580,-0.058593,0.000000
278684,1.0,0.072125,0.182623,0.350826,0.892825,0.115999,-0.435614,-1.186743,1.276407,0.394155,0.000000,0.000000,0.0,1.896291,0.117158,0.482848,-0.129133,-0.555497
34691,0.0,-0.136069,-0.046043,-0.373872,0.571904,-0.198602,0.458930,0.216813,0.968708,-0.227958,0.000000,1.958233,0.0,1.896291,-0.684266,-0.517152,-0.058593,-0.319608
378162,1.0,-0.255178,-0.297510,0.436901,0.142295,0.376597,0.205218,-0.047016,0.000000,0.014208,0.000000,0.000000,0.0,1.923970,0.426428,0.000000,-0.186769,0.569471
249800,0.0,-0.388561,-0.508799,-0.357073,-0.042688,-0.517904,-0.106751,0.281234,0.000000,-0.374509,1.067475,1.203876,0.0,-0.268383,-0.684266,0.000000,0.374848,2.861557



Scaling completed successfully.
StandardScaler, MinMaxScaler, and RobustScaler outputs are ready.


In [28]:
# ============================================================
# CELL 22 - OUTLIER DETECTION
# ============================================================

print("OUTLIER DETECTION")
print("=" * 70)

# Columns where outlier analysis is useful
outlier_cols = [
    "trip_duration",
    "trip_distance_km",
    "manhattan_distance_km",
    "passenger_count",
    "avg_duration_by_hour",
    "avg_duration_by_dayofweek",
    "avg_duration_by_month"
]

# Keep only existing columns
outlier_cols = [col for col in outlier_cols if col in train_df.columns]

outlier_summary = []

for col in outlier_cols:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outlier_count = train_df[
        (train_df[col] < lower_bound) | 
        (train_df[col] > upper_bound)
    ].shape[0]
    
    outlier_percentage = (outlier_count / train_df.shape[0]) * 100
    
    outlier_summary.append({
        "Column": col,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outlier_count,
        "Outlier Percentage": round(outlier_percentage, 2)
    })

outlier_summary_df = pd.DataFrame(outlier_summary)

display(outlier_summary_df)

print("\nOutlier detection completed successfully.")
print("Note: Outliers are detected for analysis. They should not be removed blindly.")

OUTLIER DETECTION


,Column,Lower Bound,Upper Bound,Outlier Count,Outlier Percentage
0,trip_duration,-621.500000,2094.500000,29577,5.07
1,trip_distance_km,-2.742124,7.854469,56220,9.64
2,manhattan_distance_km,-3.952520,11.350290,56232,9.64
3,passenger_count,-0.500000,3.500000,61643,10.57
4,avg_duration_by_hour,785.180625,1107.390414,13168,2.26
5,avg_duration_by_dayofweek,753.980238,1131.811246,0,0.00
6,avg_duration_by_month,829.915023,1093.189426,0,0.00



Outlier detection completed successfully.
Note: Outliers are detected for analysis. They should not be removed blindly.


In [29]:
# ============================================================
# CELL 23 - OUTLIER CAPPING
# ============================================================

print("OUTLIER CAPPING")
print("=" * 70)

# Columns selected for capping
# We cap only feature columns here, not the target directly for model input
capping_cols = [
    "trip_distance_km",
    "manhattan_distance_km",
    "passenger_count",
    "avg_duration_by_hour",
    "avg_duration_by_dayofweek",
    "avg_duration_by_month",
    "avg_duration_by_pickup_cluster",
    "avg_duration_by_dropoff_cluster"
]

# Keep only columns that exist
capping_cols = [col for col in capping_cols if col in train_df.columns and col in test_df.columns]

capping_summary = []

for col in capping_cols:
    Q1 = train_df[col].quantile(0.25)
    Q3 = train_df[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    capped_col = col + "_capped"
    
    train_df[capped_col] = train_df[col].clip(lower=lower_bound, upper=upper_bound)
    test_df[capped_col] = test_df[col].clip(lower=lower_bound, upper=upper_bound)
    
    capping_summary.append({
        "Column": col,
        "Capped Column": capped_col,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound
    })

capping_summary_df = pd.DataFrame(capping_summary)

display(capping_summary_df)

print("\nOutlier capping completed successfully.")
print("Original columns are preserved. Capped versions are created separately.")

OUTLIER CAPPING


,Column,Capped Column,Lower Bound,Upper Bound
0,trip_distance_km,trip_distance_km_capped,-2.742124,7.854469
1,manhattan_distance_km,manhattan_distance_km_capped,-3.952520,11.350290
2,passenger_count,passenger_count_capped,-0.500000,3.500000
3,avg_duration_by_hour,avg_duration_by_hour_capped,785.180625,1107.390414
4,avg_duration_by_dayofweek,avg_duration_by_dayofweek_capped,753.980238,1131.811246
5,avg_duration_by_month,avg_duration_by_month_capped,829.915023,1093.189426
6,avg_duration_by_pickup_cluster,avg_duration_by_pickup_cluster_capped,701.334653,1042.259124
7,avg_duration_by_dropoff_cluster,avg_duration_by_dropoff_cluster_capped,486.591157,1290.102621



Outlier capping completed successfully.
Original columns are preserved. Capped versions are created separately.


In [30]:
# ============================================================
# CELL 24 - TARGET TRANSFORMATION
# ============================================================

print("TARGET TRANSFORMATION")
print("=" * 70)

# Original target distribution is right-skewed
# Log transformation makes the regression target more stable
train_df["trip_duration_log"] = np.log1p(train_df["trip_duration"])
test_df["trip_duration_log"] = np.log1p(test_df["trip_duration"])

target_comparison = train_df[["trip_duration", "trip_duration_log"]].head()

display(target_comparison)

print("\nTarget transformation completed successfully.")
print("Original target: trip_duration")
print("Model target to use later: trip_duration_log")

TARGET TRANSFORMATION


,trip_duration,trip_duration_log
199266,526,6.267201
278684,1085,6.990257
34691,442,6.093570
378162,678,6.520621
249800,323,5.780744



Target transformation completed successfully.
Original target: trip_duration
Model target to use later: trip_duration_log


In [31]:
# ============================================================
# CELL 25 - LEAKAGE DETECTION
# ============================================================

print("LEAKAGE DETECTION")
print("=" * 70)

# Columns that should NOT be used as input features for model building
# because they directly or indirectly use the target or future information
leakage_cols = [
    "dropoff_datetime",
    "trip_duration",
    "trip_duration_log",
    "trip_duration_minutes",
    "trip_duration_hours",
    "average_speed_kmh",
    "speed_category",
    "duration_category",
    "trip_duration_capped",
    "trip_duration_zscore"
]

# Keep only columns that actually exist
existing_leakage_cols = [col for col in leakage_cols if col in train_df.columns]

print("Potential leakage columns found:")
print(existing_leakage_cols)

# Check columns containing suspicious target-related keywords
suspicious_keywords = [
    "duration",
    "speed",
    "dropoff"
]

suspicious_cols = []

for col in train_df.columns:
    for keyword in suspicious_keywords:
        if keyword in col.lower():
            suspicious_cols.append(col)

suspicious_cols = sorted(list(set(suspicious_cols)))

print("\nColumns needing review before model building:")
print(suspicious_cols)

print("\nLeakage detection completed successfully.")
print("These columns must be excluded from model input features.")

LEAKAGE DETECTION
Potential leakage columns found:
['dropoff_datetime', 'trip_duration', 'trip_duration_log', 'trip_duration_minutes', 'average_speed_kmh', 'speed_category']

Columns needing review before model building:
['average_speed_kmh', 'avg_duration_by_dayofweek', 'avg_duration_by_dayofweek_capped', 'avg_duration_by_dropoff_cluster', 'avg_duration_by_dropoff_cluster_capped', 'avg_duration_by_hour', 'avg_duration_by_hour_capped', 'avg_duration_by_month', 'avg_duration_by_month_capped', 'avg_duration_by_pickup_cluster', 'avg_duration_by_pickup_cluster_capped', 'dropoff_airport_distance_km', 'dropoff_cluster', 'dropoff_datetime', 'dropoff_latitude', 'dropoff_longitude', 'is_dropoff_near_airport', 'speed_category', 'trip_duration', 'trip_duration_log', 'trip_duration_minutes']

Leakage detection completed successfully.
These columns must be excluded from model input features.


In [32]:
# ============================================================
# CELL 26 - FINAL FEATURE LIST FOR MODEL BUILDING
# ============================================================

print("FINAL FEATURE LIST FOR MODEL BUILDING")
print("=" * 70)

# Target for regression model
target_col = "trip_duration_log"

# Columns that must not be used as input features
leakage_cols = [
    "id",
    "pickup_datetime",
    "dropoff_datetime",
    "trip_duration",
    "trip_duration_log",
    "trip_duration_minutes",
    "trip_duration_hours",
    "average_speed_kmh",
    "speed_category",
    "duration_category",
    "trip_duration_capped",
    "trip_duration_zscore"
]

# Raw categorical columns already encoded
raw_categorical_cols = [
    "store_and_fwd_flag",
    "pickup_day_name",
    "time_slot",
    "distance_category",
    "airport_trip_type"
]

# Columns not useful directly for model input
non_model_cols = leakage_cols + raw_categorical_cols

# Final model features = all numeric columns except leakage / non-model columns
final_feature_cols = [
    col for col in train_df.columns
    if col not in non_model_cols
    and pd.api.types.is_numeric_dtype(train_df[col])
]

print("Target column:")
print(target_col)

print("\nNumber of final features selected:")
print(len(final_feature_cols))

print("\nFinal feature columns:")
for col in final_feature_cols:
    print("-", col)

# Final X and y
X_train_final = train_df[final_feature_cols].copy()
X_test_final = test_df[final_feature_cols].copy()

y_train_final = train_df[target_col].copy()
y_test_final = test_df[target_col].copy()

print("\nFinal train/test shapes:")
print("X_train_final:", X_train_final.shape)
print("X_test_final :", X_test_final.shape)
print("y_train_final:", y_train_final.shape)
print("y_test_final :", y_test_final.shape)

print("\nFinal leakage-safe feature set created successfully.")

FINAL FEATURE LIST FOR MODEL BUILDING
Target column:
trip_duration_log

Number of final features selected:
53

Final feature columns:
- vendor_id
- passenger_count
- pickup_longitude
- pickup_latitude
- dropoff_longitude
- dropoff_latitude
- pickup_hour
- pickup_day
- pickup_month
- pickup_year
- pickup_dayofweek
- is_weekend
- is_rush_hour
- is_office_hour
- trip_distance_km
- lat_diff
- lon_diff
- manhattan_distance_km
- direction_angle
- pickup_airport_distance_km
- dropoff_airport_distance_km
- is_pickup_near_airport
- is_dropoff_near_airport
- is_morning_peak
- is_evening_peak
- is_late_night
- is_business_day
- is_month_start
- is_month_end
- distance_rush_interaction
- passenger_distance_interaction
- distance_late_night_interaction
- distance_weekend_interaction
- airport_rush_hour_interaction
- pickup_cluster
- dropoff_cluster
- avg_duration_by_hour
- avg_duration_by_dayofweek
- avg_duration_by_month
- avg_duration_by_pickup_cluster
- avg_duration_by_dropoff_cluster
- store_an

In [33]:
# ============================================================
# CELL 27 - FINAL MISSING VALUE CHECK BEFORE MODELING
# ============================================================

print("FINAL MISSING VALUE CHECK BEFORE MODELING")
print("=" * 70)

# Check missing values in final train/test feature sets
train_missing = X_train_final.isnull().sum()
test_missing = X_test_final.isnull().sum()

train_missing = train_missing[train_missing > 0].sort_values(ascending=False)
test_missing = test_missing[test_missing > 0].sort_values(ascending=False)

print("Missing values in X_train_final:")
if len(train_missing) == 0:
    print("No missing values found in X_train_final.")
else:
    display(train_missing)

print("\nMissing values in X_test_final:")
if len(test_missing) == 0:
    print("No missing values found in X_test_final.")
else:
    display(test_missing)

# Final check for infinite values
train_inf_count = np.isinf(X_train_final.select_dtypes(include=[np.number])).sum().sum()
test_inf_count = np.isinf(X_test_final.select_dtypes(include=[np.number])).sum().sum()

print("\nInfinite values check:")
print("Infinite values in X_train_final:", train_inf_count)
print("Infinite values in X_test_final :", test_inf_count)

print("\nFinal missing/infinite value check completed successfully.")

FINAL MISSING VALUE CHECK BEFORE MODELING
Missing values in X_train_final:
No missing values found in X_train_final.

Missing values in X_test_final:
No missing values found in X_test_final.

Infinite values check:
Infinite values in X_train_final: 0
Infinite values in X_test_final : 0

Final missing/infinite value check completed successfully.


In [34]:
# ============================================================
# CELL 28 - FINAL DATASET SUMMARY BEFORE MODEL BUILDING
# ============================================================

print("FINAL DATASET SUMMARY BEFORE MODEL BUILDING")
print("=" * 70)

summary_data = {
    "Item": [
        "Problem Type",
        "Original Target",
        "Final Model Target",
        "Training Rows",
        "Testing Rows",
        "Final Feature Count",
        "Train Feature Shape",
        "Test Feature Shape",
        "Missing Values in Train",
        "Missing Values in Test",
        "Infinite Values in Train",
        "Infinite Values in Test"
    ],
    "Value": [
        "Regression",
        "trip_duration",
        "trip_duration_log",
        X_train_final.shape[0],
        X_test_final.shape[0],
        X_train_final.shape[1],
        X_train_final.shape,
        X_test_final.shape,
        X_train_final.isnull().sum().sum(),
        X_test_final.isnull().sum().sum(),
        np.isinf(X_train_final.select_dtypes(include=[np.number])).sum().sum(),
        np.isinf(X_test_final.select_dtypes(include=[np.number])).sum().sum()
    ]
}

final_summary_df = pd.DataFrame(summary_data)

display(final_summary_df)

print("\nFinal feature engineering summary:")
print("- Regression problem confirmed.")
print("- Target column is trip_duration.")
print("- Log-transformed target trip_duration_log is prepared for model building.")
print("- Leakage columns are excluded from input features.")
print("- Final X_train_final, X_test_final, y_train_final, and y_test_final are ready.")
print("- Dataset is ready for the model-building notebook.")

FINAL DATASET SUMMARY BEFORE MODEL BUILDING


,Item,Value
0,Problem Type,Regression
1,Original Target,trip_duration
2,Final Model Target,trip_duration_log
3,Training Rows,583431
4,Testing Rows,145858
5,Final Feature Count,53
6,Train Feature Shape,"(583431, 53)"
7,Test Feature Shape,"(145858, 53)"
8,Missing Values in Train,0
9,Missing Values in Test,0



Final feature engineering summary:
- Regression problem confirmed.
- Target column is trip_duration.
- Log-transformed target trip_duration_log is prepared for model building.
- Leakage columns are excluded from input features.
- Final X_train_final, X_test_final, y_train_final, and y_test_final are ready.
- Dataset is ready for the model-building notebook.


In [35]:
# ============================================================
# CELL 29 - ROBUST SCALING
# ============================================================

print("ROBUST SCALING")
print("=" * 70)

# Create compatibility variables for later feature-selection cells
X_train_encoded = X_train_final.copy()
X_test_encoded = X_test_final.copy()

y_train_fe = y_train_final.copy()
y_test_fe = y_test_final.copy()

# Keep only numeric columns
numeric_encoded_features = X_train_encoded.select_dtypes(include=[np.number]).columns.tolist()

# Replace infinite values and fill missing values using train median
X_train_encoded = X_train_encoded.replace([np.inf, -np.inf], np.nan)
X_test_encoded = X_test_encoded.replace([np.inf, -np.inf], np.nan)

for col in numeric_encoded_features:
    median_value = X_train_encoded[col].median()
    X_train_encoded[col] = X_train_encoded[col].fillna(median_value)
    X_test_encoded[col] = X_test_encoded[col].fillna(median_value)

# Robust scaling
robust_scaler = RobustScaler()

X_train_robust = X_train_encoded.copy()
X_test_robust = X_test_encoded.copy()

X_train_robust[numeric_encoded_features] = robust_scaler.fit_transform(
    X_train_robust[numeric_encoded_features]
)

X_test_robust[numeric_encoded_features] = robust_scaler.transform(
    X_test_robust[numeric_encoded_features]
)

# Standard scaling for Lasso and Ridge
standard_scaler = StandardScaler()

X_train_standard = X_train_encoded.copy()
X_test_standard = X_test_encoded.copy()

X_train_standard[numeric_encoded_features] = standard_scaler.fit_transform(
    X_train_standard[numeric_encoded_features]
)

X_test_standard[numeric_encoded_features] = standard_scaler.transform(
    X_test_standard[numeric_encoded_features]
)

print("Robust Scaling completed.")
print("Standard Scaling dataset also created for Lasso and Ridge.")
print("X_train_encoded shape:", X_train_encoded.shape)
print("X_test_encoded shape :", X_test_encoded.shape)
print("X_train_standard shape:", X_train_standard.shape)

display(X_train_robust.head())

ROBUST SCALING
Robust Scaling completed.
Standard Scaling dataset also created for Lasso and Ridge.
X_train_encoded shape: (583431, 53)
X_test_encoded shape : (145858, 53)
X_train_standard shape: (583431, 53)


,vendor_id,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,pickup_hour,pickup_day,pickup_month,pickup_year,pickup_dayofweek,is_weekend,is_rush_hour,is_office_hour,trip_distance_km,lat_diff,lon_diff,manhattan_distance_km,direction_angle,pickup_airport_distance_km,dropoff_airport_distance_km,is_pickup_near_airport,is_dropoff_near_airport,is_morning_peak,is_evening_peak,is_late_night,is_business_day,is_month_start,is_month_end,distance_rush_interaction,passenger_distance_interaction,distance_late_night_interaction,distance_weekend_interaction,airport_rush_hour_interaction,pickup_cluster,dropoff_cluster,avg_duration_by_hour,avg_duration_by_dayofweek,avg_duration_by_month,avg_duration_by_pickup_cluster,avg_duration_by_dropoff_cluster,store_and_fwd_flag_encoded,pickup_day_name_encoded,time_slot_encoded,airport_trip_type_encoded,trip_distance_km_capped,manhattan_distance_km_capped,passenger_count_capped,avg_duration_by_hour_capped,avg_duration_by_dayofweek_capped,avg_duration_by_month_capped,avg_duration_by_pickup_cluster_capped,avg_duration_by_dropoff_cluster_capped
199266,0.0,0.0,-0.338212,-0.642928,0.042410,0.018178,0.4,-0.933333,-1.000000,0.0,0.25,0.0,1.0,1.0,0.160021,0.754269,0.401453,0.144992,0.338221,0.538485,-0.095600,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.406312,-0.056101,0.000000,0.000000,0.0,0.363636,-0.1,0.507382,0.190323,-0.422580,-0.058593,0.000000,0.0,-0.75,-0.5,0.0,0.160021,0.144992,0.0,0.507382,0.190323,-0.422580,-0.058593,0.000000
278684,-1.0,1.0,0.454998,0.268236,1.168828,0.525008,0.2,1.066667,0.333333,0.0,-0.50,0.0,1.0,1.0,0.072125,0.350826,0.892825,0.182623,0.115999,-0.435614,-1.186743,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.276407,0.394155,0.000000,0.000000,0.0,0.727273,0.1,1.896291,0.117158,0.482848,-0.129133,-0.555497,0.0,0.50,-1.0,0.0,0.072125,0.182623,1.0,1.896291,0.117158,0.482848,-0.129133,-0.555497
34691,0.0,0.0,-0.361881,-0.464022,0.173960,-0.735351,0.2,0.733333,-0.333333,0.0,0.50,1.0,1.0,1.0,-0.136069,-0.373872,0.571904,-0.046043,-0.198602,0.458930,0.216813,0.0,0.0,0.0,1.0,0.0,-1.0,0.0,0.0,0.968708,-0.227958,0.000000,1.958233,0.0,0.363636,0.6,1.896291,-0.684266,-0.517152,-0.058593,-0.319608,0.0,-0.25,-1.0,0.0,-0.136069,-0.046043,0.0,1.896291,-0.684266,-0.517152,-0.058593,-0.319608
378162,0.0,1.0,-0.230769,-0.110646,-0.095624,0.247827,0.1,-0.533333,0.000000,0.0,0.00,0.0,0.0,1.0,-0.255178,0.436901,0.142295,-0.297510,0.376597,0.205218,-0.047016,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.014208,0.000000,0.000000,0.0,0.000000,-0.8,1.923970,0.426428,0.000000,-0.186769,0.569471,0.0,0.25,-1.0,0.0,-0.255178,-0.297510,1.0,1.923970,0.426428,0.000000,-0.186769,0.569471
249800,-1.0,0.0,2.874805,-1.762691,2.433009,-1.911708,-1.1,0.066667,0.000000,0.0,0.50,1.0,0.0,0.0,-0.388561,-0.357073,-0.042688,-0.508799,-0.517904,-0.106751,0.281234,0.0,0.0,0.0,0.0,1.0,-1.0,0.0,0.0,0.000000,-0.374509,1.067475,1.203876,0.0,-0.727273,-1.0,-0.268383,-0.684266,0.000000,0.374848,2.861557,0.0,-0.25,0.5,0.0,-0.388561,-0.508799,0.0,-0.268383,-0.684266,0.000000,0.374848,2.069471


In [36]:
# ============================================================
# CELL 30 - FEATURE SELECTION: CORRELATION FILTERING
# ============================================================

print("FEATURE SELECTION: CORRELATION FILTERING")
print("=" * 70)

corr_data = X_train_encoded.copy()
corr_data["target"] = y_train_fe.values

target_corr = corr_data.corr(numeric_only=True)["target"].sort_values(ascending=False)

print("Correlation with Target:")
display(target_corr)

low_corr_features = target_corr[
    target_corr.abs() < 0.01
].index.tolist()

if "target" in low_corr_features:
    low_corr_features.remove("target")

print("\nLow correlation features:")
print(low_corr_features)

print("\nCorrelation filtering completed successfully.")

FEATURE SELECTION: CORRELATION FILTERING
Correlation with Target:


target                                    1.000000
trip_distance_km_capped                   0.711051
manhattan_distance_km_capped              0.703502
manhattan_distance_km                     0.555070
trip_distance_km                          0.554435
passenger_distance_interaction            0.417638
distance_rush_interaction                 0.304157
avg_duration_by_dropoff_cluster           0.299157
avg_duration_by_dropoff_cluster_capped    0.297496
avg_duration_by_pickup_cluster            0.288723
is_pickup_near_airport                    0.277384
avg_duration_by_pickup_cluster_capped     0.225348
distance_late_night_interaction           0.201323
airport_rush_hour_interaction             0.199392
distance_weekend_interaction              0.195078
is_dropoff_near_airport                   0.170201
pickup_longitude                          0.104009
avg_duration_by_hour                      0.087077
avg_duration_by_hour_capped               0.084563
is_office_hour                 


Low correlation features:
['lat_diff', 'is_rush_hour', 'is_month_end', 'is_month_start', 'dropoff_airport_distance_km', 'is_morning_peak']

Correlation filtering completed successfully.


In [37]:
# ============================================================
# CELL 31 - FEATURE SELECTION: SELECTKBEST
# ============================================================

print("FEATURE SELECTION: SELECTKBEST")
print("=" * 70)

k_value = min(15, X_train_encoded.shape[1])

selector = SelectKBest(score_func=f_regression, k=k_value)

selector.fit(X_train_encoded, y_train_fe)

selected_features_kbest = X_train_encoded.columns[
    selector.get_support()
].tolist()

kbest_scores = pd.DataFrame({
    "Feature": X_train_encoded.columns,
    "Score": selector.scores_
}).sort_values(by="Score", ascending=False)

print("Top selected features using SelectKBest:")
print(selected_features_kbest)

display(kbest_scores.head(20))

FEATURE SELECTION: SELECTKBEST
Top selected features using SelectKBest:
['trip_distance_km', 'manhattan_distance_km', 'is_pickup_near_airport', 'distance_rush_interaction', 'passenger_distance_interaction', 'distance_late_night_interaction', 'distance_weekend_interaction', 'airport_rush_hour_interaction', 'avg_duration_by_pickup_cluster', 'avg_duration_by_dropoff_cluster', 'airport_trip_type_encoded', 'trip_distance_km_capped', 'manhattan_distance_km_capped', 'avg_duration_by_pickup_cluster_capped', 'avg_duration_by_dropoff_cluster_capped']


,Feature,Score
45,trip_distance_km_capped,596630.843274
46,manhattan_distance_km_capped,571681.232697
17,manhattan_distance_km,259801.226591
14,trip_distance_km,258943.798582
30,passenger_distance_interaction,123262.454587
44,airport_trip_type_encoded,83963.006518
29,distance_rush_interaction,59475.944532
40,avg_duration_by_dropoff_cluster,57346.207640
52,avg_duration_by_dropoff_cluster_capped,56649.395837
39,avg_duration_by_pickup_cluster,53058.365098


In [38]:
# ============================================================
# CELL 32 - FEATURE SELECTION: L1 REGULARIZATION
# ============================================================

print("FEATURE SELECTION: L1 REGULARIZATION")
print("=" * 70)

lasso = Lasso(alpha=0.001, random_state=42, max_iter=10000)

lasso.fit(X_train_standard, y_train_fe)

lasso_importance = pd.DataFrame({
    "Feature": X_train_standard.columns,
    "Coefficient": lasso.coef_
})

lasso_selected = lasso_importance[
    lasso_importance["Coefficient"] != 0
].copy()

lasso_selected["Absolute Coefficient"] = lasso_selected["Coefficient"].abs()

lasso_selected = lasso_selected.sort_values(
    by="Absolute Coefficient",
    ascending=False
)

print("Features selected by L1 Regularization:")
display(lasso_selected.head(25))

FEATURE SELECTION: L1 REGULARIZATION
Features selected by L1 Regularization:


,Feature,Coefficient,Absolute Coefficient
45,trip_distance_km_capped,0.594879,0.594879
21,is_pickup_near_airport,-0.093262,0.093262
40,avg_duration_by_dropoff_cluster,-0.092356,0.092356
17,manhattan_distance_km,0.070935,0.070935
36,avg_duration_by_hour,0.067590,0.067590
5,dropoff_latitude,-0.050088,0.050088
22,is_dropoff_near_airport,-0.049282,0.049282
52,avg_duration_by_dropoff_cluster_capped,0.045919,0.045919
37,avg_duration_by_dayofweek,0.045549,0.045549
13,is_office_hour,0.043765,0.043765


In [39]:
# ============================================================
# CELL 33 - FEATURE SELECTION: L2 REGULARIZATION
# ============================================================

print("FEATURE SELECTION: L2 REGULARIZATION")
print("=" * 70)

ridge = Ridge(alpha=1.0, random_state=42)

ridge.fit(X_train_standard, y_train_fe)

ridge_importance = pd.DataFrame({
    "Feature": X_train_standard.columns,
    "Coefficient": ridge.coef_
})

ridge_importance["Absolute Coefficient"] = ridge_importance["Coefficient"].abs()

ridge_importance = ridge_importance.sort_values(
    by="Absolute Coefficient",
    ascending=False
)

print("Feature importance using L2 Regularization:")
display(ridge_importance.head(20))

FEATURE SELECTION: L2 REGULARIZATION
Feature importance using L2 Regularization:


,Feature,Coefficient,Absolute Coefficient
45,trip_distance_km_capped,0.686791,0.686791
36,avg_duration_by_hour,0.289380,0.289380
48,avg_duration_by_hour_capped,-0.238235,0.238235
17,manhattan_distance_km,0.183981,0.183981
40,avg_duration_by_dropoff_cluster,-0.112905,0.112905
21,is_pickup_near_airport,-0.105765,0.105765
14,trip_distance_km,-0.103466,0.103466
46,manhattan_distance_km_capped,-0.092978,0.092978
52,avg_duration_by_dropoff_cluster_capped,0.064024,0.064024
22,is_dropoff_near_airport,-0.061956,0.061956


In [40]:
# ============================================================
# CELL 34 - FEATURE SELECTION: RANDOM FOREST IMPORTANCE
# ============================================================

print("FEATURE SELECTION: RANDOM FOREST IMPORTANCE")
print("=" * 70)

rf = RandomForestRegressor(
    n_estimators=50,
    random_state=42,
    n_jobs=-1
)

sample_size = min(50000, len(X_train_encoded))

sample_X = X_train_encoded.sample(sample_size, random_state=42)
sample_y = y_train_fe.loc[sample_X.index]

rf.fit(sample_X, sample_y)

rf_importance = pd.DataFrame({
    "Feature": sample_X.columns,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)

print("Top Random Forest Feature Importances:")
display(rf_importance.head(20))

FEATURE SELECTION: RANDOM FOREST IMPORTANCE
Top Random Forest Feature Importances:


,Feature,Importance
14,trip_distance_km,0.321576
45,trip_distance_km_capped,0.289872
5,dropoff_latitude,0.033397
18,direction_angle,0.027258
30,passenger_distance_interaction,0.026474
16,lon_diff,0.024314
6,pickup_hour,0.021666
3,pickup_latitude,0.017151
4,dropoff_longitude,0.016593
15,lat_diff,0.016301


In [41]:
# ============================================================
# CELL 35 - SHAP FEATURE SELECTION NOTE
# ============================================================

print("""
SHAP NOTE
============================================================

SHAP is a model explainability technique.
It requires a trained model.

Since full model building is not the current goal,
SHAP should be applied later in the Model Building notebook.

Recommended later:
1. Train best model
2. Apply SHAP Explainer
3. Analyze feature contribution
4. Remove weak or harmful features if needed
""")


SHAP NOTE

SHAP is a model explainability technique.
It requires a trained model.

Since full model building is not the current goal,
SHAP should be applied later in the Model Building notebook.

Recommended later:
1. Train best model
2. Apply SHAP Explainer
3. Analyze feature contribution
4. Remove weak or harmful features if needed



In [42]:
# ============================================================
# CELL 36 - FINAL SELECTED FEATURES
# ============================================================

print("FINAL SELECTED FEATURES")
print("=" * 70)

# Combine features selected by SelectKBest and Random Forest importance
final_selected_features = list(
    set(selected_features_kbest)
    .union(set(rf_importance.head(20)["Feature"].tolist()))
)

# Keep only features that actually exist in encoded train/test data
final_selected_features = [
    col for col in final_selected_features
    if col in X_train_encoded.columns and col in X_test_encoded.columns
]

final_selected_features = sorted(final_selected_features)

print("Final selected features:")
for col in final_selected_features:
    print("-", col)

print("\nNumber of final selected features:", len(final_selected_features))

# Do not overwrite X_train_final and X_test_final from Cell 26
X_train_selected = X_train_encoded[final_selected_features].copy()
X_test_selected = X_test_encoded[final_selected_features].copy()

print("\nX_train_selected shape:", X_train_selected.shape)
print("X_test_selected shape :", X_test_selected.shape)

print("\nFinal selected feature dataset created successfully.")
print("Note: X_train_final and X_test_final from Cell 26 are preserved.")

FINAL SELECTED FEATURES
Final selected features:
- airport_rush_hour_interaction
- airport_trip_type_encoded
- avg_duration_by_dayofweek
- avg_duration_by_dropoff_cluster
- avg_duration_by_dropoff_cluster_capped
- avg_duration_by_pickup_cluster
- avg_duration_by_pickup_cluster_capped
- direction_angle
- distance_late_night_interaction
- distance_rush_interaction
- distance_weekend_interaction
- dropoff_airport_distance_km
- dropoff_latitude
- dropoff_longitude
- is_office_hour
- is_pickup_near_airport
- lat_diff
- lon_diff
- manhattan_distance_km
- manhattan_distance_km_capped
- passenger_distance_interaction
- pickup_airport_distance_km
- pickup_day
- pickup_hour
- pickup_latitude
- pickup_longitude
- trip_distance_km
- trip_distance_km_capped
- vendor_id

Number of final selected features: 29

X_train_selected shape: (583431, 29)
X_test_selected shape : (145858, 29)

Final selected feature dataset created successfully.
Note: X_train_final and X_test_final from Cell 26 are preserved.


In [45]:
# ============================================================
# CELL 37 - SAVE FINAL FEATURE DATASETS
# ============================================================

print("SAVE FINAL FEATURE DATASETS")
print("=" * 70)

from pathlib import Path

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Save selected feature datasets
X_train_selected.to_csv(output_dir / "X_train_selected.csv", index=False)
X_test_selected.to_csv(output_dir / "X_test_selected.csv", index=False)

# Save target datasets
y_train_fe.to_csv(output_dir / "y_train.csv", index=False)
y_test_fe.to_csv(output_dir / "y_test.csv", index=False)

# Save final feature list
final_feature_list = pd.DataFrame({
    "Final_Selected_Features": final_selected_features
})

final_feature_list.to_csv(output_dir / "final_selected_features.csv", index=False)

print("Files saved successfully in outputs folder:")
print("- X_train_selected.csv")
print("- X_test_selected.csv")
print("- y_train.csv")
print("- y_test.csv")
print("- final_selected_features.csv")

SAVE FINAL FEATURE DATASETS
Files saved successfully in outputs folder:
- X_train_selected.csv
- X_test_selected.csv
- y_train.csv
- y_test.csv
- final_selected_features.csv


In [46]:
# ============================================================
# CELL 38 - FINAL FEATURE ENGINEERING SUMMARY
# ============================================================

print("FINAL FEATURE ENGINEERING SUMMARY")
print("=" * 70)

summary_table = pd.DataFrame({
    "Item": [
        "Original dataset rows",
        "Original dataset columns",
        "Target column",
        "Problem type",
        "Train selected shape",
        "Test selected shape",
        "Number of selected features",
        "Leakage handled",
        "Next step"
    ],
    "Value": [
        df.shape[0],
        df.shape[1],
        "trip_duration",
        "Regression",
        str(X_train_selected.shape),
        str(X_test_selected.shape),
        len(final_selected_features),
        "dropoff_datetime and target-derived columns removed",
        "Model building"
    ]
})

display(summary_table)

print("\nFinal selected features:")
for feature in final_selected_features:
    print("-", feature)

print("\nFeature engineering completed successfully.")
print("The next notebook should start model building using X_train_selected, X_test_selected, y_train, and y_test.")

FINAL FEATURE ENGINEERING SUMMARY


,Item,Value
0,Original dataset rows,729322
1,Original dataset columns,11
2,Target column,trip_duration
3,Problem type,Regression
4,Train selected shape,"(583431, 29)"
5,Test selected shape,"(145858, 29)"
6,Number of selected features,29
7,Leakage handled,dropoff_datetime and target-derived columns re...
8,Next step,Model building



Final selected features:
- airport_rush_hour_interaction
- airport_trip_type_encoded
- avg_duration_by_dayofweek
- avg_duration_by_dropoff_cluster
- avg_duration_by_dropoff_cluster_capped
- avg_duration_by_pickup_cluster
- avg_duration_by_pickup_cluster_capped
- direction_angle
- distance_late_night_interaction
- distance_rush_interaction
- distance_weekend_interaction
- dropoff_airport_distance_km
- dropoff_latitude
- dropoff_longitude
- is_office_hour
- is_pickup_near_airport
- lat_diff
- lon_diff
- manhattan_distance_km
- manhattan_distance_km_capped
- passenger_distance_interaction
- pickup_airport_distance_km
- pickup_day
- pickup_hour
- pickup_latitude
- pickup_longitude
- trip_distance_km
- trip_distance_km_capped
- vendor_id

Feature engineering completed successfully.
The next notebook should start model building using X_train_selected, X_test_selected, y_train, and y_test.


In [44]:
# ============================================================
# CELL 39 - GENERATE FEATURE ENGINEERING WORD REPORT
# ============================================================

import os
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT

# Create reports folder
os.makedirs("../reports", exist_ok=True)

# Create Word document
doc = Document()

# ============================================================
# TITLE
# ============================================================

title = doc.add_heading("Feature Engineering Report", level=1)
title.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

subtitle = doc.add_heading("NYC Taxi Trip Duration Dataset", level=2)
subtitle.alignment = WD_PARAGRAPH_ALIGNMENT.CENTER

# ============================================================
# 1. PROJECT OVERVIEW
# ============================================================

doc.add_heading("1. Project Overview", level=2)

doc.add_paragraph(
    "This report presents the feature engineering process performed on the "
    "NYC Taxi Trip Duration dataset. The purpose of feature engineering is "
    "to transform raw data into meaningful features that improve machine "
    "learning performance."
)

doc.add_paragraph(
    "The workflow includes datetime feature extraction, interaction features, "
    "aggregation features, encoding, scaling, outlier handling, feature selection, "
    "and leakage detection."
)

# ============================================================
# 2. INITIAL DATA CHECK
# ============================================================

doc.add_heading("2. Initial Data Check", level=2)

initial_table = doc.add_table(rows=1, cols=2)
initial_table.style = "Table Grid"

hdr = initial_table.rows[0].cells
hdr[0].text = "Check"
hdr[1].text = "Finding"

initial_rows = [
    ("Dataset Shape", f"{df.shape[0]:,} rows and {df.shape[1]} columns"),
    ("Missing Values", f"{df.isnull().sum().sum()} missing values found"),
    ("Duplicate Rows", f"{df.duplicated().sum()} duplicate rows found"),
    ("Invalid passenger_count", f"{df[df['passenger_count'] <= 0].shape[0]} invalid rows"),
    ("Invalid trip_duration", f"{df[df['trip_duration'] <= 0].shape[0]} invalid rows")
]

for check, finding in initial_rows:
    row = initial_table.add_row().cells
    row[0].text = check
    row[1].text = finding

doc.add_paragraph(
    "Inference: The dataset is mostly clean. Invalid rows were removed before "
    "feature engineering."
)

# ============================================================
# 3. TRAIN TEST SPLIT
# ============================================================

doc.add_heading("3. Train-Test Split", level=2)

doc.add_paragraph(
    "The dataset was split into training and testing sets before encoding, scaling, "
    "and transformations. This prevents data leakage and ensures proper machine "
    "learning workflow."
)

split_table = doc.add_table(rows=1, cols=2)
split_table.style = "Table Grid"

hdr = split_table.rows[0].cells
hdr[0].text = "Dataset"
hdr[1].text = "Shape"

split_rows = [
    ("X_train", str(X_train.shape)),
    ("X_test", str(X_test.shape)),
    ("y_train", str(y_train.shape)),
    ("y_test", str(y_test.shape))
]

for name, shape in split_rows:
    row = split_table.add_row().cells
    row[0].text = name
    row[1].text = shape

# ============================================================
# 4. DATETIME FEATURE CREATION
# ============================================================

doc.add_heading("4. Datetime Feature Creation", level=2)

datetime_features = [
    "pickup_hour",
    "pickup_day",
    "pickup_month",
    "pickup_year",
    "pickup_dayofweek",
    "pickup_day_name",
    "is_weekend",
    "is_rush_hour",
    "time_slot"
]

doc.add_paragraph(
    "Datetime columns were used to create time-based features."
)

datetime_table = doc.add_table(rows=1, cols=2)
datetime_table.style = "Table Grid"

hdr = datetime_table.rows[0].cells
hdr[0].text = "Feature"
hdr[1].text = "Purpose"

datetime_rows = [
    ("pickup_hour", "Capture hourly traffic patterns"),
    ("pickup_day", "Capture daily variation"),
    ("pickup_month", "Capture seasonal/monthly variation"),
    ("pickup_year", "Capture yearly trend"),
    ("pickup_dayofweek", "Capture weekday pattern"),
    ("pickup_day_name", "Readable weekday information"),
    ("is_weekend", "Identify weekend trips"),
    ("is_rush_hour", "Identify traffic-heavy periods"),
    ("time_slot", "Group trips into time ranges")
]

for feature, purpose in datetime_rows:
    row = datetime_table.add_row().cells
    row[0].text = feature
    row[1].text = purpose

doc.add_paragraph(
    "Inference: Time-based features are highly important because taxi demand "
    "and traffic vary significantly across hours and days."
)

# ============================================================
# 5. DISTANCE AND SPEED FEATURES
# ============================================================

doc.add_heading("5. Distance and Speed Features", level=2)

distance_table = doc.add_table(rows=1, cols=2)
distance_table.style = "Table Grid"

hdr = distance_table.rows[0].cells
hdr[0].text = "Feature"
hdr[1].text = "Purpose"

distance_rows = [
    ("trip_distance_km", "Estimate travel distance using Haversine formula"),
    ("trip_duration_minutes", "Convert target into minutes"),
    ("trip_duration_log", "Reduce target skewness"),
    ("average_speed_kmh", "Estimate average travel speed"),
    ("speed_category", "Categorize trips based on speed")
]

for feature, purpose in distance_rows:
    row = distance_table.add_row().cells
    row[0].text = feature
    row[1].text = purpose

doc.add_paragraph(
    "Inference: Raw coordinates are difficult to interpret directly. "
    "Distance and speed features provide more meaningful information."
)

# ============================================================
# 6. INTERACTION FEATURES
# ============================================================

doc.add_heading("6. Interaction Features", level=2)

interaction_table = doc.add_table(rows=1, cols=2)
interaction_table.style = "Table Grid"

hdr = interaction_table.rows[0].cells
hdr[0].text = "Feature"
hdr[1].text = "Purpose"

interaction_rows = [
    ("distance_rush_interaction", "Study effect of rush-hour on travel distance"),
    ("passenger_distance_interaction", "Study relationship between passengers and distance")
]

for feature, purpose in interaction_rows:
    row = interaction_table.add_row().cells
    row[0].text = feature
    row[1].text = purpose

doc.add_paragraph(
    "Inference: Interaction features help capture combined effects between variables."
)

# ============================================================
# 7. AGGREGATION FEATURES
# ============================================================

doc.add_heading("7. Aggregation Features", level=2)

agg_table = doc.add_table(rows=1, cols=2)
agg_table.style = "Table Grid"

hdr = agg_table.rows[0].cells
hdr[0].text = "Feature"
hdr[1].text = "Purpose"

agg_rows = [
    ("avg_duration_by_hour", "Average trip duration by pickup hour"),
    ("avg_duration_by_day", "Average trip duration by weekday"),
    ("avg_duration_by_month", "Average trip duration by month")
]

for feature, purpose in agg_rows:
    row = agg_table.add_row().cells
    row[0].text = feature
    row[1].text = purpose

doc.add_paragraph(
    "Inference: Aggregation features capture historical behavioral patterns."
)

# ============================================================
# 8. ENCODING TECHNIQUES
# ============================================================

doc.add_heading("8. Encoding Techniques", level=2)

doc.add_paragraph(
    "Categorical variables were encoded using One-Hot Encoding. "
    "Encoding converts categorical text values into machine-readable numerical format."
)

doc.add_paragraph(
    "drop_first=True was used to reduce multicollinearity."
)

# ============================================================
# 9. SCALING AND TRANSFORMATION
# ============================================================

doc.add_heading("9. Scaling and Transformation", level=2)

scaling_table = doc.add_table(rows=1, cols=2)
scaling_table.style = "Table Grid"

hdr = scaling_table.rows[0].cells
hdr[0].text = "Technique"
hdr[1].text = "Purpose"

scaling_rows = [
    ("StandardScaler", "Standardize features around mean and standard deviation"),
    ("MinMaxScaler", "Scale features between 0 and 1"),
    ("RobustScaler", "Handle scaling in presence of outliers"),
    ("Log Transformation", "Reduce skewness in target variable")
]

for technique, purpose in scaling_rows:
    row = scaling_table.add_row().cells
    row[0].text = technique
    row[1].text = purpose

doc.add_paragraph(
    "Inference: Different scaling techniques were explored to understand their "
    "impact on numerical feature distributions."
)

# ============================================================
# 10. OUTLIER HANDLING
# ============================================================

doc.add_heading("10. Outlier Handling", level=2)

outlier_table = doc.add_table(rows=1, cols=2)
outlier_table.style = "Table Grid"

hdr = outlier_table.rows[0].cells
hdr[0].text = "Technique"
hdr[1].text = "Purpose"

outlier_rows = [
    ("IQR Method", "Detect outliers using quartiles"),
    ("Z-Score Method", "Detect outliers using standard deviation"),
    ("IQR Capping", "Limit extreme values without deleting rows")
]

for technique, purpose in outlier_rows:
    row = outlier_table.add_row().cells
    row[0].text = technique
    row[1].text = purpose

doc.add_paragraph(
    "Inference: Outliers were reviewed carefully because some long trips may be valid business cases."
)

# ============================================================
# 11. FEATURE SELECTION
# ============================================================

doc.add_heading("11. Feature Selection", level=2)

selection_table = doc.add_table(rows=1, cols=2)
selection_table.style = "Table Grid"

hdr = selection_table.rows[0].cells
hdr[0].text = "Technique"
hdr[1].text = "Purpose"

selection_rows = [
    ("Correlation Filtering", "Remove weakly related features"),
    ("SelectKBest", "Select statistically important features"),
    ("L1 Regularization", "Remove less important features"),
    ("L2 Regularization", "Measure feature importance"),
    ("Random Forest Importance", "Tree-based feature importance"),
    ("SHAP Recommendation", "Future explainability analysis")
]

for technique, purpose in selection_rows:
    row = selection_table.add_row().cells
    row[0].text = technique
    row[1].text = purpose

doc.add_paragraph(
    "Inference: Multiple feature selection methods were explored to identify strong predictive features."
)

# ============================================================
# 12. LEAKAGE DETECTION
# ============================================================

doc.add_heading("12. Leakage Detection", level=2)

doc.add_paragraph(
    "Potential leakage columns were identified and removed before preparing "
    "the final modeling dataset."
)

leakage_table = doc.add_table(rows=1, cols=2)
leakage_table.style = "Table Grid"

hdr = leakage_table.rows[0].cells
hdr[0].text = "Leakage Column"
hdr[1].text = "Reason"

leakage_rows = [
    ("dropoff_datetime", "Contains future trip completion information"),
    ("trip_duration_minutes", "Derived directly from target"),
    ("trip_duration_log", "Derived directly from target"),
    ("average_speed_kmh", "Uses final trip duration"),
    ("speed_category", "Derived from final speed"),
    ("trip_duration_capped", "Derived from target"),
    ("trip_duration_zscore", "Derived from target")
]

for col, reason in leakage_rows:
    row = leakage_table.add_row().cells
    row[0].text = col
    row[1].text = reason

doc.add_paragraph(
    "Inference: Leakage detection is critical because leakage can create unrealistic model performance."
)

# ============================================================
# 13. FINAL CONCLUSION
# ============================================================

doc.add_heading("13. Final Conclusion", level=2)

doc.add_paragraph(
    "Feature engineering successfully transformed raw NYC taxi trip data into "
    "more meaningful machine learning features."
)

doc.add_paragraph(
    "Datetime, location, interaction, aggregation, scaling, encoding, and feature "
    "selection techniques were applied systematically."
)

doc.add_paragraph(
    "The final engineered dataset is now suitable for future model building."
)

# ============================================================
# FORMATTING
# ============================================================

for paragraph in doc.paragraphs:
    for run in paragraph.runs:
        run.font.name = "Calibri"
        run.font.size = Pt(11)

# ============================================================
# SAVE REPORT
# ============================================================

report_path = "../reports/NYC_Taxi_Feature_Engineering_Report.docx"

doc.save(report_path)

print("Feature Engineering Word report created successfully.")
print("Report saved at:", report_path)

Feature Engineering Word report created successfully.
Report saved at: ../reports/NYC_Taxi_Feature_Engineering_Report.docx
